[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-9_executed.ipynb)

---


# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 9: Quantum-Walk-Inspired Search

### Goal

This notebook explores an oracle-guided walk over container-to-stack assignments. It represents a search state by a complex amplitude vector, marks zero-violation assignments with a phase flip, applies a reversible permutation on the configuration graph, and samples assignments from the resulting probability distribution.

The implementation is a small, classical state-vector simulation for teaching. It is **not** a quantum circuit or a demonstrated quantum advantage: the walk operator is a simplified reversible permutation, the coin contributes only a global phase, and the simulation explicitly stores its basis states and amplitudes in classical memory.

### Inputs, outputs, and scope

- **Inputs:** container priorities, a number of stacks, a tier capacity, and walk and measurement parameters.
- **Output:** a sampled assignment with the smallest observed number of priority violations, together with measurement statistics and plots.
- **Scope:** each assignment gives one stack to each container. Containers assigned to the same stack are ordered from bottom to top by their order in the input list. The model checks stack capacity and adjacent priority conflicts in that order. It does not model crane travel, relocation costs, equipment, time windows, or a complete terminal-yard operating plan.

The container list in the example is ordered by increasing priority number, where a smaller number means higher retrieval priority. Under the stated bottom-to-top convention, a lower-priority container above a higher-priority one is a conflict. The oracle marks assignments with no such conflicts; it does not distinguish among feasible assignments by relocation cost.

### Method at a glance

1. Enumerate capacity-feasible stack assignments and place them in a uniform superposition.
2. Apply a phase oracle that changes the sign of each zero-violation assignment's amplitude.
3. Apply a reversible walk operator that swaps paired neighboring configurations and leaves unpaired configurations fixed.
4. Repeat the oracle and walk, then draw samples according to the final measurement probabilities.

### What the results can and cannot show

The plots show how the simulated probability distribution and sampled violations change during these specified updates. They do not establish convergence, a quantum speedup, hardware feasibility, or the behavior of a physical quantum walk. Because the walk is a permutation and the phase oracle changes phases rather than probabilities, this particular update rule does not amplify marked-state probabilities.

## Mathematical model

Let the containers be indexed by \(i \in \{0,\ldots,n-1\}\), the stacks by \(s \in \{0,\ldots,S-1\}\), and the tier capacity be \(T\). An assignment is a vector \(x=(x_0,\ldots,x_{n-1})\), where \(x_i=s\) means container \(i\) is placed in stack \(s\). The enumerated state set is

$$\mathcal{X}=\left\{x\in\{0,\ldots,S-1\}^n:\sum_{i=0}^{n-1}\mathbf{1}[x_i=s]\leq T\text{ for every }s\right\}.$$

Here, \(\mathbf{1}[\cdot]\) equals 1 when its condition is true and 0 otherwise. A stack's bottom-to-top order is the order of its container indices. If \(p_i\) is container \(i\)'s priority, an adjacent conflict occurs when \(p_i<p_j\) for consecutive containers \(i,j\) in the same stack: the higher-priority container is below the lower-priority one. The violation count \(v(x)\) is the number of such adjacent conflicts. The oracle marks the set \(\mathcal{G}=\{x\in\mathcal{X}:v(x)=0\}\).

The algorithm stores an amplitude \(\psi_t(x)\in\mathbb{C}\) for each enumerated assignment. The initial state is the uniform superposition, and the phase oracle is

$$\psi_0(x)=\frac{1}{\sqrt{|\mathcal{X}|}},\qquad
(O\psi)(x)=(-1)^{\mathbf{1}[x\in\mathcal{G}]}\psi(x).$$

For the walk, the implementation constructs a deterministic pairing of neighboring configurations. The resulting operator \(S\) swaps the two states in every pair and fixes any unpaired state. Since \(S^2=I\), it is a permutation operator and is unitary. The configured coin phase is \(C=e^{i\pi/d}I\), where \(d\) is `coin_dimension`; it is a global phase, not a coin register. One update is

$$\psi_{t+1}=\mathcal{N}\left(\alpha S O\psi_t+(1-\alpha)\eta_t\right),\qquad
\alpha=e^{-\gamma},$$

where \(\gamma\) is the decoherence rate, \(\eta_t\) is the configured complex Gaussian noise, and \(\mathcal{N}\) normalizes the state to unit norm. With zero noise, the update is a unitary permutation and phase operation up to a global phase; with noise, it is a phenomenological, normalized simulation rather than a unitary quantum evolution. The measurement probability is \(|\psi_t(x)|^2\).

The recurrence is implemented by `apply_quality_oracle` and `quantum_walk_step`; `optimize` applies them in that order. For \(N=|\mathcal{X}|\), storing the state and constructing its pairing require memory proportional to \(N\). Pairing checks up to \(n(S-1)\) candidate neighbors per state, so its straightforward construction takes \(O(NnS)\) time. Oracle evaluation per iteration is also proportional to the number of enumerated states, apart from the work needed to count violations. The state space itself can grow exponentially with the number of containers.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
import itertools
import math
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container data used by the assignment model."""

    id: str
    priority: int
    current_stack: int
    current_tier: int


@dataclass
class QuantumState:
    """Classically stored amplitudes over a finite assignment basis."""

    amplitudes: np.ndarray
    basis_states: List[Tuple[int, ...]]

    def normalize(self) -> None:
        """Keep the total measurement probability equal to one."""
        norm = np.linalg.norm(self.amplitudes)
        if norm > 0:
            self.amplitudes /= norm

    def get_probability_distribution(self) -> np.ndarray:
        """Convert amplitudes into measurement probabilities."""
        return np.abs(self.amplitudes) ** 2


@dataclass
class QuantumWalkConfig:
    """Parameters for the illustrative state-vector walk."""

    num_containers: int
    num_stacks: int
    num_tiers: int
    walk_steps: int = 15
    coin_dimension: int = 2
    measurement_samples: int = 500
    noise_rate: float = 0.01
    decoherence_rate: float = 0.001


class QuantumQualityOracle:
    """Mark assignments with no adjacent priority conflicts."""

    def __init__(self, containers: List[Container], stacks: int, tiers: int):
        self.containers = containers
        self.stacks = stacks
        self.tiers = tiers
        self.container_priorities = {container.id: container.priority for container in containers}

    def is_valid_configuration(self, stack_assignments: Tuple[int, ...]) -> bool:
        """Check assignment length, stack labels, and tier capacity."""
        if len(stack_assignments) != len(self.containers):
            return False
        if any(stack < 0 or stack >= self.stacks for stack in stack_assignments):
            return False

        stack_counts = defaultdict(int)
        for stack in stack_assignments:
            stack_counts[stack] += 1
            if stack_counts[stack] > self.tiers:
                return False
        return True

    def count_violations(self, stack_assignments: Tuple[int, ...]) -> int:
        """Count adjacent retrieval-priority conflicts within stacks."""
        if len(stack_assignments) != len(self.containers):
            raise ValueError("The assignment must include one stack for each container.")

        stack_containers = defaultdict(list)
        for index, stack in enumerate(stack_assignments):
            if stack < 0 or stack >= self.stacks:
                raise ValueError("Every stack assignment must be a valid stack index.")
            container = self.containers[index]
            stack_containers[stack].append(
                (self.container_priorities[container.id], index)
            )

        violations = 0
        for stack in stack_containers.values():
            stack.sort(key=lambda item: item[1])
            priorities = [priority for priority, _ in stack]
            for lower_tier_priority, upper_tier_priority in zip(priorities, priorities[1:]):
                if lower_tier_priority < upper_tier_priority:
                    violations += 1
        return violations

    def is_optimal_configuration(self, stack_assignments: Tuple[int, ...]) -> bool:
        """Return whether an assignment is capacity-feasible and unblocked."""
        return (
            self.is_valid_configuration(stack_assignments)
            and self.count_violations(stack_assignments) == 0
        )

In [ ]:
class QuantumWalkOptimizer:
    """Apply a phase oracle and a reversible matching-based walk."""

    def __init__(self, containers: List[Container], config: QuantumWalkConfig):
        if len(containers) != config.num_containers:
            raise ValueError("The container count must match the configuration.")
        if config.num_containers < 0 or config.num_stacks <= 0 or config.num_tiers <= 0:
            raise ValueError("Container count must be nonnegative; stacks and tiers must be positive.")
        if config.walk_steps < 0 or config.measurement_samples <= 0:
            raise ValueError("Walk steps must be nonnegative and measurement samples must be positive.")
        if config.coin_dimension <= 0 or config.noise_rate < 0 or config.decoherence_rate < 0:
            raise ValueError("Coin dimension must be positive and noise rates must be nonnegative.")

        self.containers = containers
        self.config = config
        self.oracle = QuantumQualityOracle(
            containers, config.num_stacks, config.num_tiers
        )
        self.quantum_state = self._initialize_quantum_state()
        self.shift_permutation = self._create_shift_permutation()
        self.coin_phase = np.exp(1j * np.pi / config.coin_dimension)
        self.optimization_history = {
            "probabilities": [],
            "best_solutions": [],
            "convergence_metrics": [],
            "oracle_calls": 0,
        }

    def _initialize_quantum_state(self) -> QuantumState:
        """Start with equal amplitude on every capacity-feasible assignment."""
        basis_states = self._generate_valid_basis_states()
        if not basis_states:
            raise ValueError("No capacity-feasible assignments exist for this instance.")

        amplitudes = np.ones(len(basis_states), dtype=complex) / np.sqrt(len(basis_states))
        state = QuantumState(amplitudes, basis_states)
        state.normalize()
        return state

    def _generate_valid_basis_states(self) -> List[Tuple[int, ...]]:
        """Enumerate capacity-feasible assignments for the configured instance."""
        stacks = range(self.config.num_stacks)
        valid_states = []
        for assignment in itertools.product(stacks, repeat=self.config.num_containers):
            if self._is_valid_assignment(assignment):
                valid_states.append(assignment)
        return valid_states

    def _is_valid_assignment(self, assignment: Tuple[int, ...]) -> bool:
        """Apply the tier-capacity constraint to one assignment."""
        stack_counts = defaultdict(int)
        for stack in assignment:
            stack_counts[stack] += 1
            if stack_counts[stack] > self.config.num_tiers:
                return False
        return True

    def _create_shift_permutation(self) -> np.ndarray:
        """Pair neighboring states to create a reversible graph walk."""
        basis_states = self.quantum_state.basis_states
        state_indices = {state: index for index, state in enumerate(basis_states)}
        permutation = np.arange(len(basis_states))
        paired = np.zeros(len(basis_states), dtype=bool)

        for index, state in enumerate(basis_states):
            if paired[index]:
                continue

            for container_index in range(self.config.num_containers):
                for stack in range(self.config.num_stacks):
                    if stack == state[container_index]:
                        continue

                    neighbor = list(state)
                    neighbor[container_index] = stack
                    neighbor = tuple(neighbor)
                    neighbor_index = state_indices.get(neighbor)
                    if neighbor_index is not None and not paired[neighbor_index]:
                        permutation[index] = neighbor_index
                        permutation[neighbor_index] = index
                        paired[index] = True
                        paired[neighbor_index] = True
                        break
                if paired[index]:
                    break

        return permutation

    def apply_quality_oracle(self, quantum_state: QuantumState) -> QuantumState:
        """Flip the phase of every zero-violation basis state."""
        marked_amplitudes = quantum_state.amplitudes.copy()
        for index, basis_state in enumerate(quantum_state.basis_states):
            self.optimization_history["oracle_calls"] += 1
            if self.oracle.is_optimal_configuration(basis_state):
                marked_amplitudes[index] *= -1
        return QuantumState(marked_amplitudes, quantum_state.basis_states)

    def quantum_walk_step(self, quantum_state: QuantumState) -> QuantumState:
        """Apply the global coin phase, reversible shift, and noise model."""
        coin_amplitudes = self.coin_phase * quantum_state.amplitudes
        shifted_amplitudes = coin_amplitudes[self.shift_permutation]

        noise = self.config.noise_rate * (
            np.random.normal(size=len(shifted_amplitudes))
            + 1j * np.random.normal(size=len(shifted_amplitudes))
        )
        decoherence_factor = np.exp(-self.config.decoherence_rate)
        final_amplitudes = (
            decoherence_factor * shifted_amplitudes
            + (1 - decoherence_factor) * noise
        )

        new_state = QuantumState(final_amplitudes, quantum_state.basis_states)
        new_state.normalize()
        return new_state

    def _calculate_entropy(self, probabilities: np.ndarray) -> float:
        """Calculate Shannon entropy in bits, omitting zero-probability states."""
        nonzero_probabilities = probabilities[probabilities > 0]
        return float(-np.sum(nonzero_probabilities * np.log2(nonzero_probabilities)))

    def _calculate_optimal_probability(self) -> float:
        """Sum the probability of all zero-violation assignments."""
        probabilities = self.quantum_state.get_probability_distribution()
        return float(sum(
            probability
            for probability, basis_state in zip(
                probabilities, self.quantum_state.basis_states
            )
            if self.oracle.is_optimal_configuration(basis_state)
        ))

    def _measure_and_analyze(self) -> Tuple[Tuple[int, ...], float, Dict[str, Any]]:
        """Sample the final state and summarize observed assignment quality."""
        probabilities = self.quantum_state.get_probability_distribution()
        measurement_indices = np.random.choice(
            len(probabilities),
            size=self.config.measurement_samples,
            p=probabilities,
        )
        measurement_results = [
            (
                self.quantum_state.basis_states[index],
                self.oracle.count_violations(self.quantum_state.basis_states[index]),
            )
            for index in measurement_indices
        ]

        best_solution, best_violations = min(measurement_results, key=lambda item: item[1])
        violation_distribution = [violations for _, violations in measurement_results]
        success_probability = sum(
            violations == 0 for violations in violation_distribution
        ) / len(violation_distribution)
        analysis = {
            "best_violations": best_violations,
            "success_probability": success_probability,
            "violation_distribution": violation_distribution,
            "avg_violations": float(np.mean(violation_distribution)),
            "violation_std": float(np.std(violation_distribution)),
            "measurement_samples": self.config.measurement_samples,
        }
        return best_solution, success_probability, analysis

    def optimize(self) -> Tuple[Tuple[int, ...], float, Dict[str, Any]]:
        """Run the configured oracle-and-walk iterations, then measure."""
        print("Starting the illustrative state-vector walk.")
        print(
            f"Problem size: {self.config.num_containers} containers, "
            f"{self.config.num_stacks} stacks"
        )
        print(f"Enumerated capacity-feasible assignments: {len(self.quantum_state.basis_states)}")
        print(f"Walk steps: {self.config.walk_steps}")

        start_time = time.time()
        initial_probabilities = self.quantum_state.get_probability_distribution()
        initial_best_index = int(np.argmax(initial_probabilities))
        initial_best = self.quantum_state.basis_states[initial_best_index]
        print(f"Initial best-state violations: {self.oracle.count_violations(initial_best)}")
        print(f"Initial probability entropy: {self._calculate_entropy(initial_probabilities):.4f} bits")

        for step in range(self.config.walk_steps):
            marked_state = self.apply_quality_oracle(self.quantum_state)
            self.quantum_state = self.quantum_walk_step(marked_state)

            probabilities = self.quantum_state.get_probability_distribution()
            best_index = int(np.argmax(probabilities))
            best_solution = self.quantum_state.basis_states[best_index]
            best_violations = self.oracle.count_violations(best_solution)
            optimal_probability = self._calculate_optimal_probability()
            entropy = self._calculate_entropy(probabilities)

            self.optimization_history["probabilities"].append(probabilities.copy())
            self.optimization_history["best_solutions"].append(
                (best_solution, best_violations)
            )
            self.optimization_history["convergence_metrics"].append(
                {
                    "step": step + 1,
                    "entropy": entropy,
                    "optimal_probability": optimal_probability,
                    "best_violations": best_violations,
                }
            )

            if (step + 1) % 3 == 0:
                print(
                    f"Step {step + 1}: best violations = {best_violations}, "
                    f"zero-violation probability = {optimal_probability:.4f}, "
                    f"entropy = {entropy:.4f} bits"
                )

        final_solution, success_probability, analysis = self._measure_and_analyze()
        print(f"Simulation completed in {time.time() - start_time:.2f} seconds")
        print(f"Best measured violations: {self.oracle.count_violations(final_solution)}")
        print(f"Measured zero-violation fraction: {success_probability:.3f}")
        print(f"Oracle evaluations: {self.optimization_history['oracle_calls']}")
        return final_solution, success_probability, analysis

In [ ]:
def visualize_quantum_optimization(
    optimization_history: Dict[str, Any], config: QuantumWalkConfig
) -> None:
    """Plot the simulated probability, entropy, and violation history."""
    metrics = optimization_history["convergence_metrics"]
    if not metrics:
        print("No walk steps were recorded, so there are no history plots to display.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Illustrative State-Vector Walk: Simulation History", fontsize=16)
    steps = [metric["step"] for metric in metrics]

    axes[0, 0].plot(
        steps,
        [metric["optimal_probability"] for metric in metrics],
        marker="o",
    )
    axes[0, 0].set_xlabel("Walk step")
    axes[0, 0].set_ylabel("Probability of zero-violation assignments")
    axes[0, 0].set_title("Marked-State Probability")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(steps, [metric["entropy"] for metric in metrics], color="tab:red", marker="s")
    axes[0, 1].set_xlabel("Walk step")
    axes[0, 1].set_ylabel("Shannon entropy (bits)")
    axes[0, 1].set_title("Probability-Distribution Entropy")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        steps,
        [metric["best_violations"] for metric in metrics],
        color="tab:green",
        marker="^",
    )
    axes[1, 0].set_xlabel("Walk step")
    axes[1, 0].set_ylabel("Violations in most probable assignment")
    axes[1, 0].set_title("Most-Probable Assignment Quality")
    axes[1, 0].grid(True, alpha=0.3)

    final_probabilities = optimization_history["probabilities"][-1]
    top_count = min(20, len(final_probabilities))
    top_indices = np.argsort(final_probabilities)[::-1][:top_count]
    top_probabilities = final_probabilities[top_indices]
    is_marked = [
        QuantumQualityOracle(
            containers, config.num_stacks, config.num_tiers
        ).is_optimal_configuration(
            optimizer.quantum_state.basis_states[index]
        )
        for index in top_indices
    ]
    colors = ["tab:green" if marked else "tab:purple" for marked in is_marked]
    axes[1, 1].bar(range(top_count), top_probabilities, color=colors)
    axes[1, 1].set_xlabel("Rank by probability")
    axes[1, 1].set_ylabel("Measurement probability")
    axes[1, 1].set_title(f"Final Distribution: Top {top_count} States")
    axes[1, 1].legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color="tab:green", label="Zero violations"),
            plt.Rectangle((0, 0), 1, 1, color="tab:purple", label="One or more violations"),
        ],
        loc="best",
    )
    axes[1, 1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def analyze_quantum_advantage(
    containers: List[Container],
    quantum_result: Tuple[Tuple[int, ...], float, Dict[str, Any]],
    classical_result: Dict[str, Any],
    num_stacks: int,
) -> Dict[str, Any]:
    """Compare observed violations without claiming quantum advantage."""
    quantum_solution, quantum_success, quantum_analysis = quantum_result
    quantum_violations = quantum_analysis["best_violations"]
    classical_violations = classical_result["violations"]

    print("\n" + "=" * 60)
    print("SOLUTION-QUALITY COMPARISON")
    print("=" * 60)
    print(f"Containers: {len(containers)}")
    print(f"Stacks: {num_stacks}")
    print(f"Best measured walk assignment: {quantum_solution}")
    print(f"Walk violations: {quantum_violations}")
    print(f"Classical greedy violations: {classical_violations}")
    print(f"Measured zero-violation fraction: {quantum_success:.3f}")
    print(
        "These results compare assignment quality for this instance; they do not "
        "measure quantum speedup or establish quantum advantage."
    )

    return {
        "quantum_violations": quantum_violations,
        "classical_violations": classical_violations,
        "success_probability": quantum_success,
        "comparison_scope": "Observed assignment quality only; no quantum advantage claim.",
    }

In [ ]:
print("=" * 60)
print("YARD PRE-MARSHALLING: ILLUSTRATIVE WALK SIMULATION")
print("=" * 60)

containers = [
    Container("A", 1, 0, 0),
    Container("B", 2, 0, 1),
    Container("C", 3, 1, 0),
    Container("D", 4, 1, 1),
    Container("E", 5, 2, 0),
    Container("F", 6, 2, 1),
    Container("G", 7, 3, 0),
    Container("H", 8, 3, 1),
]

num_stacks = 5
num_tiers = 2
print(f"\nContainers: {len(containers)}")
print(f"Stacks: {num_stacks}")
print(f"Tier capacity per stack: {num_tiers}")
print("\nContainer data (priority, current stack, current tier):")
for container in containers:
    print(
        f"  {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

total_assignments = num_stacks ** len(containers)
print(f"\nUnrestricted assignments: {total_assignments:,}")
print(
    "Capacity-feasible assignments are enumerated by the optimizer; "
    "their count is reported after initialization."
)

In [ ]:
config = QuantumWalkConfig(
    num_containers=len(containers),
    num_stacks=num_stacks,
    num_tiers=num_tiers,
    walk_steps=12,
    coin_dimension=2,
    measurement_samples=200,
    noise_rate=0.01,
    decoherence_rate=0.001,
)

print("\nSimulation parameters:")
print(f"  Walk steps: {config.walk_steps}")
print(f"  Coin phase parameter: {config.coin_dimension}")
print(f"  Measurement samples: {config.measurement_samples}")
print(f"  Noise rate: {config.noise_rate}")
print(f"  Decoherence rate: {config.decoherence_rate}")

optimizer = QuantumWalkOptimizer(containers, config)
print(f"Capacity-feasible basis states: {len(optimizer.quantum_state.basis_states):,}")
print("The full basis is stored explicitly in classical memory.")

quantum_solution, quantum_success, quantum_analysis = optimizer.optimize()
quantum_result = (quantum_solution, quantum_success, quantum_analysis)

print(f"\nBest measured assignment: {quantum_solution}")
print(f"Violations in that assignment: {quantum_analysis['best_violations']}")
print(f"Measured zero-violation fraction: {quantum_success:.3f}")

assert optimizer.oracle.is_valid_configuration(quantum_solution)
assert quantum_analysis["best_violations"] == optimizer.oracle.count_violations(
    quantum_solution
)
assert np.isclose(
    np.sum(optimizer.quantum_state.get_probability_distribution()), 1.0
)

In [ ]:
visualize_quantum_optimization(optimizer.optimization_history, config)


def classical_greedy_solution(
    containers: List[Container], stacks: int, tiers: int
) -> Tuple[Tuple[int, ...], int]:
    """Assign containers to the least-loaded stack with available capacity."""
    stack_counts = [0] * stacks
    assignment = [0] * len(containers)

    for index, _ in enumerate(containers):
        available_stacks = [
            stack for stack in range(stacks) if stack_counts[stack] < tiers
        ]
        if not available_stacks:
            raise ValueError("The total stack capacity is insufficient for all containers.")
        selected_stack = min(available_stacks, key=lambda stack: stack_counts[stack])
        assignment[index] = selected_stack
        stack_counts[selected_stack] += 1

    return tuple(assignment), len(containers)


def random_solution(
    containers: List[Container], stacks: int, tiers: int
) -> Tuple[Tuple[int, ...], int]:
    """Sample a capacity-feasible assignment by rejection sampling."""
    max_attempts = 1000
    for _ in range(max_attempts):
        assignment = tuple(
            random.randint(0, stacks - 1) for _ in containers
        )
        counts = [assignment.count(stack) for stack in range(stacks)]
        if all(count <= tiers for count in counts):
            return assignment, len(containers)

    return classical_greedy_solution(containers, stacks, tiers)


classical_solution, classical_moves = classical_greedy_solution(
    containers, num_stacks, num_tiers
)
random_assignment, random_moves = random_solution(
    containers, num_stacks, num_tiers
)
comparison_oracle = QuantumQualityOracle(containers, num_stacks, num_tiers)
classical_violations = comparison_oracle.count_violations(classical_solution)
random_violations = comparison_oracle.count_violations(random_assignment)

print("\nClassical greedy baseline:")
print(f"  Assignment: {classical_solution}")
print(f"  Violations: {classical_violations}")
print(f"  Containers assigned: {classical_moves}")
print("\nRandom feasible baseline:")
print(f"  Assignment: {random_assignment}")
print(f"  Violations: {random_violations}")
print(f"  Containers assigned: {random_moves}")

print("\n" + "=" * 60)
print("OBSERVED ASSIGNMENT-QUALITY COMPARISON")
print("=" * 60)
print(f"{'Method':<24}{'Violations':<14}{'Quality':<16}")
print("-" * 54)
for method, violations in [
    ("Illustrative walk", quantum_analysis["best_violations"]),
    ("Classical greedy", classical_violations),
    ("Random feasible", random_violations),
]:
    quality = "Zero violations" if violations == 0 else "Has violations"
    print(f"{method:<24}{violations:<14}{quality:<16}")

print(
    "This comparison concerns observed violations only; it is not a comparison "
    "of quantum and classical computational complexity."
)

In [ ]:
classical_result = {
    "violations": classical_violations,
    "moves": classical_moves,
}

comparison_analysis = analyze_quantum_advantage(
    containers,
    quantum_result,
    classical_result,
    num_stacks,
)

print("\nInterpretation:")
print(
    "The zero-violation fraction is the fraction of final samples with no "
    "modeled adjacent priority conflicts. It is an empirical fraction from the "
    "specified simulation, not a hardware success probability or a guarantee "
    "about unobserved assignments."
)

In [ ]:
def analyze_scalability(container_counts: List[int], stacks: int) -> List[Dict[str, Any]]:
    """Report unrestricted state-space growth and state-vector storage needs."""
    results = []
    for container_count in container_counts:
        total_assignments = stacks ** container_count
        results.append(
            {
                "containers": container_count,
                "unrestricted_assignments": total_assignments,
                "amplitude_entries_if_fully_enumerated": total_assignments,
                "index_bits_for_unrestricted_space": math.ceil(
                    math.log2(total_assignments)
                ),
            }
        )
    return results


container_sizes = [8, 10, 12, 15, 20]
scalability_results = analyze_scalability(container_sizes, num_stacks)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Growth of the Explicit Assignment State Space")
sizes = [result["containers"] for result in scalability_results]
space_sizes = [result["unrestricted_assignments"] for result in scalability_results]
index_bits = [result["index_bits_for_unrestricted_space"] for result in scalability_results]

axes[0].plot(sizes, space_sizes, marker="o", color="tab:blue")
axes[0].set_yscale("log")
axes[0].set_xlabel("Number of containers")
axes[0].set_ylabel("Unrestricted assignments (count, log scale)")
axes[0].set_title("Assignment-Space Growth")
axes[0].grid(True, alpha=0.3)

axes[1].plot(sizes, index_bits, marker="s", color="tab:red")
axes[1].set_xlabel("Number of containers")
axes[1].set_ylabel("Bits to index unrestricted assignments")
axes[1].set_title("Compact Index Size")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nState-space growth (unrestricted counts):")
print(f"{'Containers':<14}{'Assignments':<22}{'Index bits':<12}")
print("-" * 48)
for result in scalability_results:
    print(
        f"{result['containers']:<14}"
        f"{result['unrestricted_assignments']:<22,}"
        f"{result['index_bits_for_unrestricted_space']:<12}"
    )

print(
    "These counts describe the unrestricted assignment space, not the number "
    "of capacity-feasible states. Explicit state-vector simulation requires "
    "one amplitude per included basis state, so the feasible-state count also "
    "controls memory and computation. No speedup or hardware capacity is inferred."
)

## Interpretation and limitations

### Reading the simulation

- The **zero-violation probability** is the sum of the final measurement probabilities of assignments marked by the oracle.
- The **entropy** measures how spread out the probability distribution is. For probabilities \(q_x\), Shannon entropy is \(H(q)=-\sum_x q_x\log_2(q_x)\), with zero-probability terms omitted. Lower entropy means a more concentrated distribution; it does not by itself mean that the distribution favors feasible assignments.
- The **best measured assignment** is the sample with the fewest modeled violations. The reported fraction of zero-violation samples is an empirical statistic from the configured number of measurements.

In this implementation, a noiseless walk step permutes amplitudes and multiplies them by phases. A permutation changes which basis state carries an amplitude but does not change the set of amplitude magnitudes. The phase oracle also leaves magnitudes unchanged. Consequently, the noiseless update does not amplify the total probability of marked assignments. The configured noise can change probabilities after normalization, but such changes are a feature of the noise model, not evidence of quantum search amplification.

### Model and computational limits

The assignment model is deliberately narrow. It uses the input order as the within-stack order and counts adjacent priority conflicts; it does not optimize the number of relocations or use the containers' current positions in its objective. Capacity-feasible assignments are enumerated explicitly, and the simulation stores a complex amplitude for every enumerated assignment. The state space can therefore become too large for this classical implementation even when a compact mathematical description of an assignment exists.

A physical quantum walk requires a fully specified quantum register, unitary operators, and a circuit or other realizable implementation. The simplified global coin phase and matching-based permutation here are teaching devices for exploring amplitudes, phase marking, and measurement. The numerical experiment does not establish a quantum advantage over classical methods, predict practical hardware requirements, or support claims about future implementation timelines.